# Clustering Data for Nori | BVH vertical trajectories

This notebook adapts the MVNX hand-foot-pelvis extraction to use the transformed BVH CSVs in `bvh_to_csv_centered/`.

Important axis note: Nori's downstream code expects fields named `*_zPos`, but BVH vertical position is the **Y** axis. Therefore these `*_zPos` fields are filled from BVH `.Y` columns.

In [1]:
import os
import pickle
from datetime import datetime

import numpy as np
import pandas as pd
from scipy.io import savemat


today = datetime.now().strftime("%d%b").lower()

with open("data/selected_piece_list.pkl", "rb") as f:
    piece_list = pickle.load(f)

# Process ALL dance modes, matching the reference MVNX notebook.
modes = ["group", "individual", "audience"]

# BVH source data. BVH vertical axis is Y, but Nori's downstream code expects
# the legacy field names ending in zPos.
bvh_dir = "bvh_to_csv_centered"
bvh_suffix = "_T_worldpos_rc_both_frame_smooth0.4.csv"

hand_onset_dir = "data/hand_clap_onsets_05jun2026"
foot_onset_root = "data/dance_onsets_v4_0.007_foot_jun3"
base_path_cycles = "data/virtual_cycles"

cluster_data = {
    "file_name": [],

    "dmode_name": [],
    "dmode_seg_idx": [],
    "dmode_start": [],
    "dmode_end": [],

    "cycle_idx": [],
    "cycle_start": [],
    "cycle_end": [],

    "location": [],
    "ensemble": [],
    "day": [],
    "rec_no": [],
    "piece": [],

    # Legacy field names expected by Nori. Values come from BVH Y (vertical).
    "L_hand_zPos": [],
    "R_hand_zPos": [],
    "LR_hand_zPos": [],

    "L_foot_zPos": [],
    "R_foot_zPos": [],
    "LR_foot_zPos": [],

    "pelvis_zPos": [],

    # New onset names requested for Nori.
    "hand_onsets": [],
    "left_foot_onsets": [],
    "right_foot_onsets": [],

    "Dun": [],
    "J1": [],
    "J2": [],
}


def load_onset_csv(path, column):
    """Load onset times from a CSV; return empty array if unavailable."""
    if not os.path.exists(path):
        print(f"[WARNING] Missing onset file: {path}")
        return np.array([], dtype=float)
    try:
        return pd.read_csv(path)[column].dropna().to_numpy(dtype=float)
    except Exception as exc:
        print(f"[WARNING] Could not read onset file {path}: {exc}")
        return np.array([], dtype=float)


for file_name in piece_list:
    parts = file_name.split("_")
    location = parts[0]
    ensemble = parts[1]
    day = parts[2]
    recording_no = parts[3]
    piece = parts[4]

    bvh_path = os.path.join(bvh_dir, f"{file_name}{bvh_suffix}")
    cycles_csv = os.path.join(base_path_cycles, f"{file_name}_C.csv")
    drum_onsets_path = f"data/drum_onsets/{file_name}.csv"

    if not os.path.exists(bvh_path):
        print(f"[WARNING] Missing BVH CSV: {bvh_path}; skipping")
        continue
    if not os.path.exists(cycles_csv):
        print(f"[WARNING] Missing cycles CSV: {cycles_csv}; skipping")
        continue
    if not os.path.exists(drum_onsets_path):
        print(f"[WARNING] Missing drum onsets: {drum_onsets_path}; skipping")
        continue

    # Load BVH vertical trajectories (Y axis). Field names remain zPos for compatibility.
    bvh_df = pd.read_csv(bvh_path)
    times = bvh_df["Time"].to_numpy(dtype=float)

    pelvis_zpos = bvh_df["Hips.Y"].to_numpy(dtype=float)
    Left_foot_zpos = bvh_df["LeftAnkle.Y"].to_numpy(dtype=float)
    Right_foot_zpos = bvh_df["RightAnkle.Y"].to_numpy(dtype=float)
    Left_hand_zpos = bvh_df["LeftWrist.Y"].to_numpy(dtype=float)
    Right_hand_zpos = bvh_df["RightWrist.Y"].to_numpy(dtype=float)

    # Load hand clap onsets: one hand_onsets field only.
    hand_onsets_path = os.path.join(hand_onset_dir, f"{file_name}_hand_clap_onsets.csv")
    hand_onsets = load_onset_csv(hand_onsets_path, "contact_times")

    # Load new foot onset CSVs.
    foot_onset_dir = os.path.join(foot_onset_root, f"{file_name}_T", "onset_info")
    left_foot_onsets_all = load_onset_csv(
        os.path.join(foot_onset_dir, f"{file_name}_T_left_foot_onsets.csv"),
        "time_sec",
    )
    right_foot_onsets_all = load_onset_csv(
        os.path.join(foot_onset_dir, f"{file_name}_T_right_foot_onsets.csv"),
        "time_sec",
    )

    # Load drum onset data.
    drum_df = pd.read_csv(drum_onsets_path)
    Dun_onsets = drum_df["Dun"].dropna().to_numpy(dtype=float)
    J1_onsets = drum_df["J1"].dropna().to_numpy(dtype=float)
    J2_onsets = drum_df["J2"].dropna().to_numpy(dtype=float)

    # Load virtual cycle onsets.
    cyc_df = pd.read_csv(cycles_csv)
    onsets = cyc_df["Virtual Onset"].to_numpy(dtype=float)

    for dance_mode in modes:
        dance_mode_path = f"data/dance_modes_ts/{file_name}_{dance_mode}.pkl"

        if os.path.exists(dance_mode_path):
            with open(dance_mode_path, "rb") as f:
                dmode_ts = pickle.load(f)
        else:
            print(f"{file_name} {dance_mode} does not exist")
            continue

        for dmode_idx, dmode in enumerate(dmode_ts):
            dmode_start, dmode_end = dmode

            # Filter virtual-cycle onsets to this dance-mode segment.
            mode_mask = (onsets >= dmode_start) & (onsets <= dmode_end)
            mode_onsets = onsets[mode_mask]

            cycle_times = [
                (round(mode_onsets[i], 3), round(mode_onsets[i + 1], 3))
                for i in range(len(mode_onsets) - 1)
            ]

            for c_idx, (c_start, c_end) in enumerate(cycle_times):
                win_mask = (times >= c_start) & (times <= c_end)

                # Position trajectories ----------------------------------
                # Hand
                left_hand_zpos_win = Left_hand_zpos[win_mask]
                right_hand_zpos_win = Right_hand_zpos[win_mask]
                left_right_hand_zpos = np.concatenate((left_hand_zpos_win, right_hand_zpos_win))

                # Feet
                left_foot_zpos_win = Left_foot_zpos[win_mask]
                right_foot_zpos_win = Right_foot_zpos[win_mask]
                left_right_feet_zpos = np.concatenate((left_foot_zpos_win, right_foot_zpos_win))

                # Pelvis / Hips
                pelvis_zpos_win = pelvis_zpos[win_mask]

                # Onsets -------------------------------------------------
                c_hand_onsets = hand_onsets[
                    (hand_onsets >= c_start) & (hand_onsets <= c_end)
                ]
                c_left_foot_onsets = left_foot_onsets_all[
                    (left_foot_onsets_all >= c_start) & (left_foot_onsets_all <= c_end)
                ]
                c_right_foot_onsets = right_foot_onsets_all[
                    (right_foot_onsets_all >= c_start) & (right_foot_onsets_all <= c_end)
                ]

                # Drum onsets
                c_Dun_onset = Dun_onsets[(Dun_onsets >= c_start) & (Dun_onsets <= c_end)]
                c_J1_onset = J1_onsets[(J1_onsets >= c_start) & (J1_onsets <= c_end)]
                c_J2_onset = J2_onsets[(J2_onsets >= c_start) & (J2_onsets <= c_end)]

                # Save metadata -----------------------------------------
                cluster_data["file_name"].append(file_name)

                cluster_data["dmode_name"].append(dance_mode)
                cluster_data["dmode_seg_idx"].append(dmode_idx + 1)
                cluster_data["dmode_start"].append(dmode_start)
                cluster_data["dmode_end"].append(dmode_end)

                cluster_data["cycle_idx"].append(c_idx + 1)
                cluster_data["cycle_start"].append(c_start)
                cluster_data["cycle_end"].append(c_end)

                cluster_data["location"].append(location)
                cluster_data["ensemble"].append(ensemble)
                cluster_data["day"].append(day)
                cluster_data["rec_no"].append(recording_no)
                cluster_data["piece"].append(piece)

                # Save trajectories -------------------------------------
                cluster_data["L_hand_zPos"].append(left_hand_zpos_win)
                cluster_data["R_hand_zPos"].append(right_hand_zpos_win)
                cluster_data["LR_hand_zPos"].append(left_right_hand_zpos)

                cluster_data["L_foot_zPos"].append(left_foot_zpos_win)
                cluster_data["R_foot_zPos"].append(right_foot_zpos_win)
                cluster_data["LR_foot_zPos"].append(left_right_feet_zpos)

                cluster_data["pelvis_zPos"].append(pelvis_zpos_win)

                # Save onsets -------------------------------------------
                cluster_data["hand_onsets"].append(c_hand_onsets)
                cluster_data["left_foot_onsets"].append(c_left_foot_onsets)
                cluster_data["right_foot_onsets"].append(c_right_foot_onsets)

                cluster_data["Dun"].append(c_Dun_onset)
                cluster_data["J1"].append(c_J1_onset)
                cluster_data["J2"].append(c_J2_onset)


out_dir = os.path.join("data", "cluster_data", "Nori")
os.makedirs(out_dir, exist_ok=True)

save_pkl = os.path.join(
    out_dir,
    f"cluster_data_Nori_bvh_all_dmodes_hand_feet_pelvis_with_onsets_{today}.pkl",
)
save_mat = os.path.join(
    out_dir,
    f"cluster_data_Nori_bvh_all_dmodes_hand_feet_pelvis_with_onsets_{today}.mat",
)

with open(save_pkl, "wb") as f:
    pickle.dump(cluster_data, f)

savemat(save_mat, cluster_data)

print(f"Saved pkl: {save_pkl}")
print(f"Saved mat: {save_mat}")
print(f"Total cycles: {len(cluster_data['file_name'])}")
print(f"Fields: {list(cluster_data.keys())}")

BKO_E1_D1_07_Suku group does not exist
BKO_E1_D1_07_Suku audience does not exist
BKO_E1_D1_08_Suku group does not exist
BKO_E1_D1_08_Suku audience does not exist
BKO_E1_D5_01_Maraka group does not exist
BKO_E1_D5_01_Maraka audience does not exist
BKO_E1_D5_04_Suku group does not exist
BKO_E1_D5_04_Suku audience does not exist
BKO_E2_D3_11_Suku group does not exist
BKO_E2_D3_11_Suku audience does not exist
BKO_E2_D3_13_Suku group does not exist
BKO_E2_D3_13_Suku audience does not exist
Saved pkl: data/cluster_data/Nori/cluster_data_Nori_bvh_all_dmodes_hand_feet_pelvis_with_onsets_05jun.pkl
Saved mat: data/cluster_data/Nori/cluster_data_Nori_bvh_all_dmodes_hand_feet_pelvis_with_onsets_05jun.mat
Total cycles: 5533
Fields: ['file_name', 'dmode_name', 'dmode_seg_idx', 'dmode_start', 'dmode_end', 'cycle_idx', 'cycle_start', 'cycle_end', 'location', 'ensemble', 'day', 'rec_no', 'piece', 'L_hand_zPos', 'R_hand_zPos', 'LR_hand_zPos', 'L_foot_zPos', 'R_foot_zPos', 'LR_foot_zPos', 'pelvis_zPos', 